<a href="https://colab.research.google.com/github/RJRM999/EDA-DS-Projects/blob/main/Web_Research_Agent(ReAct_Pattern).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai tavily-python python-dotenv

In [ ]:
from openai import OpenAI
from tavily import TavilyClient
import textwrap

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")

Enter OpenAI API Key: ··········
Enter Tavily API Key: ··········


In [ ]:
llm_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [ ]:
class WebResearchAgent:
    def __init__(self, topic):
        self.topic = topic
        self.questions = []
        self.search_results = {}
        self.report = ""

    def generate_research_questions(self):
        """
        Planning Phase:
        Uses an LLM to generate 5-6 research questions for the topic.
        """

        prompt = f"""
        You are a research planning assistant.

        Generate 5 to 6 well-structured research questions for the topic:
        "{self.topic}"

        The questions should cover:
        - background
        - current trends
        - benefits
        - challenges
        - real-world use cases
        - future outlook

        Return only the questions as a numbered list.
        """

        response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You generate clear research questions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.4
        )

        output = response.choices[0].message.content

        self.questions = [
            q.strip()
            for q in output.split("\n")
            if q.strip()
        ]

        return self.questions

    def search_web(self):
        """
        Acting Phase:
        Uses Tavily to search the web for each research question.
        """

        for question in self.questions:
            clean_question = question.split(".", 1)[-1].strip()

            results = tavily_client.search(
                query=clean_question,
                search_depth="advanced",
                max_results=3
            )

            extracted_results = []

            for item in results.get("results", []):
                extracted_results.append({
                    "title": item.get("title", ""),
                    "url": item.get("url", ""),
                    "content": item.get("content", "")
                })

            self.search_results[question] = extracted_results

        return self.search_results

    def generate_report(self):
        """
        Report Generation Phase:
        Uses the gathered search results to produce a structured report.
        """

        research_context = ""

        for question, results in self.search_results.items():
            research_context += f"\n\nResearch Question: {question}\n"

            for idx, result in enumerate(results, start=1):
                research_context += f"""
                Source {idx}:
                Title: {result['title']}
                URL: {result['url']}
                Content: {result['content']}
                """

        prompt = f"""
        You are a research analyst.

        Create a structured research report on the topic:
        "{self.topic}"

        Use the research information below:

        {research_context}

        The report must include:
        1. Title
        2. Introduction
        3. Separate sections for each research question
        4. Key findings
        5. Conclusion
        6. References with source URLs

        Write clearly and professionally.
        """

        response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You write structured research reports."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        self.report = response.choices[0].message.content

        return self.report

    def run(self):
        print("Planning research questions...")
        self.generate_research_questions()

        print("Searching the web...")
        self.search_web()

        print("Generating final report...")
        final_report = self.generate_report()

        return final_report

In [ ]:
topic = input("Enter your researh topic: ")

agent = WebResearchAgent(topic)
final_report = agent.run()

print(final_report)

Enter your researh topic: Climate Change
Planning research questions...
Searching the web...
Generating final report...
# Research Report on Climate Change

## Introduction
Climate change represents one of the most pressing global challenges of our time, characterized by long-term alterations in temperature, precipitation patterns, and extreme weather events. This report explores the historical and scientific foundations of climate change, current trends, potential benefits of addressing the issue, challenges faced in mitigation and adaptation, successful integration of climate considerations into real-world applications, and projected future scenarios based on varying levels of global action.

## 1. Historical and Scientific Foundations of Climate Change
The understanding of climate change has evolved significantly over the past two centuries. Early experiments in the 1800s suggested that human-produced carbon dioxide (CO2) could insulate the Earth’s atmosphere. By the late 1950s, CO2

In [ ]:
with open("final_research_report.md", "w", encoding="utf-8") as file:
    file.write(final_report)

print("Report saved as final_research_report.md")

Report saved as final_research_report.md


In [ ]:
print("Generated Research Questions:\n")

for q in agent.questions:
    print(q)

Generated Research Questions:

1. What are the historical and scientific foundations of climate change, and how have they shaped current understanding of the phenomenon?
2. What are the most significant current trends in climate change, including temperature increases, sea-level rise, and extreme weather events, and how do these trends vary across different regions?
3. What are the potential benefits of addressing climate change through renewable energy adoption, sustainable practices, and policy interventions for both the environment and the economy?
4. What are the main challenges faced by governments, organizations, and communities in mitigating climate change and adapting to its impacts, particularly in vulnerable regions?
5. How have real-world use cases, such as urban planning, agriculture, and disaster management, successfully integrated climate change considerations to enhance resilience and sustainability?
6. What is the projected future outlook for climate change over the nex

In [ ]:
for question, results in agent.search_results.items():
    print("\n" + "="*80)
    print(question)
    print("="*80)

    for result in results:
        print("Title:", result["title"])
        print("URL:", result["url"])
        print("Content:", result["content"][:500])
        print("-"*80)


1. What are the historical and scientific foundations of climate change, and how have they shaped current understanding of the phenomenon?
Title: Climate Change History - Timeline, Events & Earth | HISTORY
URL: https://www.history.com/articles/history-of-climate-change
Content: 13

Sources

Climate change is the long-term alteration in Earth’s climate and weather patterns. It took nearly a century of research and data to convince the vast majority of the scientific community that human activity could alter the climate of our entire planet. In the 1800s, experiments suggesting that human-produced carbon dioxide (CO2) and other gases could collect in the atmosphere and insulate Earth were met with more curiosity than concern. By the late 1950s, CO2 readings would offer s
--------------------------------------------------------------------------------
Title: A brief history of climate change discoveries
URL: https://www.discover.ukri.org/a-brief-history-of-climate-change-discoveries/inde

## Explanation of the ReAct Agent

This project implements a Web Research Agent using the ReAct pattern, which combines reasoning and acting.

### Reasoning / Planning Phase

The agent first uses a Large Language Model to understand the user-provided topic and generate 5 to 6 research questions. This represents the reasoning step because the LLM decides what aspects of the topic should be investigated.

### Acting Phase

After generating the research questions, the agent uses the Tavily web search tool to search the internet for each question. The agent collects titles, URLs, and short content snippets from the search results.

### Report Generation Phase

Finally, the agent sends the collected information back to the LLM and asks it to create a structured report. The report contains a title, introduction, separate sections for each research question, key findings, conclusion, and references.

### Design Patterns Used

1. Planning Pattern: The agent generates research questions before searching.
2. Tool-Use Pattern: The agent uses external tools such as an LLM and Tavily Search.
3. ReAct Pattern: The agent reasons about the topic, acts by searching the web, and produces a final answer.